# Session 40: Hyperparameter Tuning II — Random Forest Regressor
**Week 4 Randomized Search Optimization for Ensemble Models**

In this notebook, we tune a Random Forest Regressor using 5-fold cross-validated Randomized Search (`RandomizedSearchCV`). We evaluate how searching over tree count (`n_estimators`), maximum tree depth (`max_depth`), and minimum split threshold (`min_samples_split`) optimizes testing set generalization relative to default ensemble settings.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Resolve project root directory
PROJECT_ROOT = Path.cwd().resolve()
for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (parent / ".git").exists():
        PROJECT_ROOT = parent
        break

DATA_DIRECTORY = PROJECT_ROOT / "data"
REPORTS_DIRECTORY = PROJECT_ROOT / "reports" / "tables"
REPORTS_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Project Root:", PROJECT_ROOT)

Project Root: /home/nikhil/Desktop/VSCode/GSSRP/student-performance-prediction-ml


In [2]:
def load_regression_split():
    # Attempt to locate processed student dataset
    candidates = list((DATA_DIRECTORY / "processed").rglob("*.parquet")) + list((DATA_DIRECTORY / "processed").rglob("*.csv"))
    for path in candidates:
        if any(term in path.name.lower() for term in ["comparison", "prediction", "result"]):
            continue
        try:
            table = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
            if "G3" in table.columns:
                # Exclude G1 and G2 to preserve early-warning research design
                drop_cols = [c for c in ["G1", "G2", "G3"] if c in table.columns]
                X = table.drop(columns=drop_cols).copy()
                X = pd.get_dummies(X, drop_first=True, dtype=float)
                y = table["G3"]
                
                Xtr, Xte, ytr, yte = train_test_split(
                    X, y, test_size=0.20, random_state=42
                )
                return Xtr, Xte, ytr, yte
        except Exception:
            continue
    raise FileNotFoundError("Could not locate processed regression datasets.")

Xtr_f, Xte_f, ytr, yte = load_regression_split()
print(f"Data Loaded — Training Features: {Xtr_f.shape}, Test Features: {Xte_f.shape}")

def eval_reg(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mse)),
        "R2": float(r2_score(y_true, y_pred))
    }

Data Loaded — Training Features: (316, 39), Test Features: (79, 39)


In [3]:
# Instantiate default Random Forest Regressor baseline
default_rf = RandomForestRegressor(random_state=42)
default_rf.fit(Xtr_f, ytr)

default_preds = default_rf.predict(Xte_f)
default_metrics = eval_reg(yte, default_preds)

print("Default Random Forest Regressor Performance (Test Split):")
print("-" * 50)
for metric_name, val in default_metrics.items():
    print(f"{metric_name}: {val:.4f}")

Default Random Forest Regressor Performance (Test Split):
--------------------------------------------------
MAE: 3.1477
RMSE: 3.9354
R2: 0.2447


In [4]:
# Define hyperparameter search space matching Session 40 configuration
param_space = {
    "n_estimators": [200, 300, 500],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10]
}

# Instantiate 5-fold cross-validated RandomizedSearchCV (8 iterations)
random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_space,
    n_iter=8,
    cv=5,
    scoring="r2",
    random_state=42,
    n_jobs=-1,
    return_train_score=True
)

random_search.fit(Xtr_f, ytr)

print("RandomizedSearchCV Optimization Results:")
print("-" * 50)
print(f"Best Hyperparameters Found: {random_search.best_params_}")
print(f"Best 5-Fold Cross-Validated R2 Score: {random_search.best_score_:.4f}")

RandomizedSearchCV Optimization Results:
--------------------------------------------------
Best Hyperparameters Found: {'n_estimators': 500, 'min_samples_split': 10, 'max_depth': None}
Best 5-Fold Cross-Validated R2 Score: 0.2688


In [5]:
# Evaluate optimal estimator on untouched test split
best_rf = random_search.best_estimator_
tuned_preds = best_rf.predict(Xte_f)
tuned_metrics = eval_reg(yte, tuned_preds)

# Construct side-by-side metric comparison table
rf_comparison_df = pd.DataFrame([
    {"Model": "Default Random Forest", **default_metrics},
    {"Model": "Tuned Random Forest (RandomizedSearch)", **tuned_metrics}
])

print("Untuned vs. Tuned Performance Comparison:")
display(rf_comparison_df.round(4))

# Export local artifact table
output_csv_path = REPORTS_DIRECTORY / "session40_rf_tuning_results.csv"
rf_comparison_df.to_csv(output_csv_path, index=False)
print(f"Saved results table to: {output_csv_path}")

Untuned vs. Tuned Performance Comparison:


,Model,MAE,RMSE,R2
0,Default Random Forest,3.1477,3.9354,0.2447
1,Tuned Random Forest (RandomizedSearch),3.0836,3.8412,0.2804


Saved results table to: /home/nikhil/Desktop/VSCode/GSSRP/student-performance-prediction-ml/reports/tables/session40_rf_tuning_results.csv


In [6]:
# Validation assertions
assert hasattr(random_search, "best_params_"), "RandomizedSearchCV fit failed!"
assert random_search.n_iter == 8, "n_iter parameter mismatch!"
assert len(tuned_preds) == len(yte), "Prediction output length mismatch!"
assert np.isfinite(tuned_preds).all(), "Predictions contain non-finite values!"

print("=" * 72)
print("SESSION 40 HYPERPARAMETER TUNING COMPLETED SUCCESSFULLY")
print("=" * 72)

SESSION 40 HYPERPARAMETER TUNING COMPLETED SUCCESSFULLY
